In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
(PROJECT_ROOT / "reports").mkdir(exist_ok=True)


# Transfer learning: spatial nested grid search

Run after `preprocessing_age_labels.ipynb` and `baseline.ipynb`. The Madrid source
forest is frozen. We search target-forest hyperparameters and blend weights using
only each permitted Amsterdam adaptation sample. Separate outer spatial folds
measure the complete selection procedure.

**Evaluation boundary:** outer folds were viewed during earlier experiments.
These are development results, not untouched organizer-test scores. Never use
these outer scores to choose a different grid or a fixed winning configuration.


In [ ]:
from scripts.verify_grid_transfer import verify
from scripts.grid_transfer import run_experiment, FOREST_GRID, SOURCE_WEIGHTS
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'pyproject.toml').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUTPUT = ROOT / 'data/transfer_grid_search'
# Set True to rerun all 75 adaptation searches (several minutes).
# Existing results are verified below; missing results trigger a full run.
RUN_GRID_SEARCH = False
print('Forest grid:', FOREST_GRID)
print('Madrid blend weights:', SOURCE_WEIGHTS)

## Fit and select

The exhaustive `ParameterGrid` search evaluates 32 combinations over three
spatial inner folds (two only when necessary for class-complete training).
It reuses forest probabilities across blend weights to avoid redundant fits.
150 trees, `balanced_subsample`, and all 54 baseline predictors remain fixed.

Highest mean inner macro F1 determines the configuration separately for each
training sample. Exact ties use grid order, which places the baseline first.
Median imputation fits inside each training pipeline. Outer-fold locations and
blocks never enter adaptation or selection. Three seeded sampling repetitions
match the baseline exactly, at 5/25/50/100/200 labels per class.


In [ ]:
if RUN_GRID_SEARCH or not (OUTPUT / 'manifest.json').is_file():
    run_experiment()
verification = verify()

## Held-out development comparison

Mean scores first average sampling repetitions within each fold; ± is sample SD
across five spatial folds, not a confidence interval. A zero Madrid weight means
inner validation selected target-only prediction; it is not evidence of a transfer gain.


In [ ]:
summary = pd.read_csv(OUTPUT / 'summary.csv')
paired = pd.read_csv(OUTPUT / 'paired_fold_differences.csv')
selections = pd.read_csv(OUTPUT / 'selected_parameters.csv')
display(summary.round(4))
display(paired.groupby('samples_per_class').delta_macro_f1.agg(
    ['mean', 'min', 'max']).round(4))
for n, rows in selections.groupby('samples_per_class'):
    print(f'Labels/class={n}: inner-selected configurations (15 runs)')
    display(rows.groupby(['max_depth', 'max_features', 'min_samples_leaf', 'source_weight'], dropna=False)
            .size().sort_values(ascending=False).rename('selected_runs').reset_index())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for strategy, group in summary.groupby('strategy'):
    ax.errorbar(group.samples_per_class, group.macro_f1_mean,
                yerr=group.macro_f1_fold_sd, marker='o', capsize=3,
                label=strategy.replace('_', ' '))
ax.set(xlabel='Amsterdam labels per class', ylabel='Outer spatial macro F1',
       title='Nested grid search versus fixed transfer blend')
ax.legend()
ax.grid(alpha=.2)
fig.tight_layout()
# Keep the exported plot outside the immutable experiment artifact manifest.
fig.savefig(ROOT / 'reports/transfer_grid_search.png', dpi=160)
plt.show()

## Class-level results and limitations

All per-class precision/recall/F1 scores are exported. Inspect both a low-label
budget (25/class) and the largest budget (200/class). Blocks can touch and remain
spatially correlated. City-specific class boundaries still need confirmation.
The configurations are optimal only within this finite grid and each training
sample; no universal optimum is established. Freeze the procedure before an
untouched organizer-test evaluation.


In [ ]:
class_summary = pd.read_csv(OUTPUT / 'class_summary.csv')
display(
    class_summary.loc[class_summary.samples_per_class.isin([25, 200])].round(4))
print((ROOT / 'reports/TRANSFER_GRID_SEARCH_RESULTS.md').read_text())

Method reference: [scikit-learn nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html).
